# S6E4 Predicting Irrigation Need

## Score: .95787

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", 200)

In [4]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "playground-series-s6e4"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = PROJECT_ROOT / "submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
SEED = 42
N_SPLITS = 3

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Expected data files under {DATA_DIR.resolve()}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()

train shape: (630000, 21)
test shape : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [5]:
display(train_df[TARGET].value_counts(dropna=False))
print("\nTarget distribution (%):")
display((train_df[TARGET].value_counts(normalize=True) * 100).round(2))

missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

print("\nTop missing-rate features (train):")
display(missing_train.head(10))
print("\nTop missing-rate features (test):")
display(missing_test.head(10))

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


Target distribution (%):


Irrigation_Need
Low       58.72
Medium    37.95
High       3.33
Name: proportion, dtype: float64


Top missing-rate features (train):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64


Top missing-rate features (test):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64

In [6]:
y = train_df[TARGET].copy()
X = train_df.drop(columns=[TARGET]).copy()
X_test = test_df.copy()

feature_cols = [c for c in X.columns if c != ID_COL]
X = X[feature_cols]
X_test = X_test[feature_cols]

cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical   : {len(cat_cols)} -> {cat_cols}")
print(f"Numerical     : {len(num_cols)}")

Total features: 19
Categorical   : 8 -> ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
Numerical     : 11


In [7]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

classes = sorted(y.unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

oof_pred = np.empty(len(X), dtype=object)
test_proba = np.zeros((len(X_test), len(classes)), dtype=np.float64)
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.07,
        depth=7,
        loss_function="MultiClass",
        eval_metric="TotalF1",
        random_seed=SEED + fold,
        verbose=100,
        thread_count=-1,
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        early_stopping_rounds=60,
    )

    valid_pred = model.predict(X_valid).reshape(-1)
    oof_pred[valid_idx] = valid_pred

    fold_score = balanced_accuracy_score(y_valid, valid_pred)
    fold_scores.append(fold_score)
    print(f"Fold {fold}: balanced_accuracy = {fold_score:.6f}")

    test_proba += model.predict_proba(X_test)

cv_score = balanced_accuracy_score(y, oof_pred)
print("\nCV balanced accuracy (OOF):", round(cv_score, 6))
print("Fold scores:", [round(s, 6) for s in fold_scores])
print("Mean fold score:", round(float(np.mean(fold_scores)), 6))

0:	learn: 0.9803579	test: 0.9806620	best: 0.9806620 (0)	total: 1.47s	remaining: 12m 11s
100:	learn: 0.9849063	test: 0.9849496	best: 0.9849954 (69)	total: 2m 25s	remaining: 9m 34s
200:	learn: 0.9850718	test: 0.9850234	best: 0.9850234 (200)	total: 4m 19s	remaining: 6m 25s
300:	learn: 0.9851709	test: 0.9850482	best: 0.9850527 (264)	total: 6m 28s	remaining: 4m 16s
400:	learn: 0.9852842	test: 0.9850673	best: 0.9850721 (347)	total: 8m 47s	remaining: 2m 10s
Stopped by overfitting detector  (60 iterations wait)

bestTest = 0.9850958779
bestIteration = 413

Shrink model to first 414 iterations.
Fold 1: balanced_accuracy = 0.960862
0:	learn: 0.9807797	test: 0.9805018	best: 0.9805018 (0)	total: 1.44s	remaining: 11m 57s
100:	learn: 0.9838313	test: 0.9834472	best: 0.9834472 (100)	total: 2m 24s	remaining: 9m 30s
200:	learn: 0.9840358	test: 0.9835775	best: 0.9835818 (184)	total: 4m 35s	remaining: 6m 50s
300:	learn: 0.9841963	test: 0.9836264	best: 0.9836312 (299)	total: 6m 48s	remaining: 4m 30s
400:	l

In [8]:
test_proba /= N_SPLITS
test_pred_idx = np.argmax(test_proba, axis=1)
test_pred = [idx_to_class[i] for i in test_pred_idx]

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred,
})
submission.to_csv(SUB_PATH, index=False)

print("Saved:", SUB_PATH.resolve())
submission.head()

Saved: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\s6e4-predicting-irrigation-need\submission_catboost_notebook.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
